#### Librerias

In [1]:
import pandas as pd
import os
import re
import unicodedata

#### Carga de datos

In [2]:
def cargar_y_concatenar(ruta_carpeta, tipo):
    dfs = []

    for archivo in os.listdir(ruta_carpeta):
        if archivo.endswith(".csv"):

            # Match robusto por tipo + año
            match = re.search(rf"{tipo}_(20\d{{2}})", archivo, re.IGNORECASE)

            if match:
                ruta = os.path.join(ruta_carpeta, archivo)
                año = int(match.group(1))

                try:
                    df = pd.read_csv(
                        ruta,
                        sep=";",
                        encoding="cp1252",
                        encoding_errors="replace",
                        engine="python",
                        on_bad_lines="skip"
                    )

                    df["año"] = año
                    dfs.append(df)

                except Exception as e:
                    print(f"[ERROR] {archivo}: {e}")

    if not dfs:
        raise ValueError(f"No se encontraron archivos para: {tipo}")

    return pd.concat(dfs, ignore_index=True)

In [3]:
ruta = "../../data"

df_viajeros = cargar_y_concatenar(ruta, "viajeros")
df_pernoctaciones = cargar_y_concatenar(ruta, "Pernoctaciones")

#### Revisión de datos

In [4]:
df_viajeros.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6900 entries, 0 to 6899
Data columns (total 5 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Comunidades y Ciudades Autónomas  6900 non-null   object
 1   País de residencia                6900 non-null   object
 2   Meses                             6900 non-null   object
 3   Total                             6900 non-null   object
 4   año                               6900 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 269.7+ KB


In [5]:
# Convertir nombres de columnas a minúsculas y eliminar espacios
df_viajeros.columns = df_viajeros.columns.str.strip().str.lower()
df_pernoctaciones.columns = df_pernoctaciones.columns.str.strip().str.lower()

In [6]:
df_pernoctaciones.columns

Index(['comunidades y ciudades autónomas', 'país de residencia', 'meses',
       'total', 'año'],
      dtype='object')

In [7]:
# Renombrar columnas para mayor claridad del dato que contienen
df_viajeros.rename(columns={
    "meses": "mes",
    "comunidades y ciudades autónomas": "ccaa",
    "país de residencia": "pais_origen",
    "total":"qty_viajeros"
}, inplace=True)

df_pernoctaciones.rename(columns={
    "meses": "mes",
    "comunidades y ciudades autónomas": "ccaa",
    "país de residencia": "pais_origen",
    "total":"qty_viajeros"
}, inplace=True)


In [8]:
df_viajeros['pais_origen'].value_counts()

pais_origen
Residentes en España    300
Polonia                 300
Estados Unidos          300
Japón                   300
Suiza                   300
Rusia                   300
Noruega                 300
Suecia                  300
República Checa         300
Reino Unido             300
Portugal                300
Países Bajos            300
Alemania                300
Luxemburgo              300
Italia                  300
Irlanda                 300
Grecia                  300
Francia                 300
Finlandia               300
Dinamarca               300
Bélgica                 300
Austria                 300
Países africanos        300
Name: count, dtype: int64

#### Transformación

In [9]:
# Convertir columnas de total a numéricas
if df_viajeros["qty_viajeros"].dtype == "object":
    df_viajeros["qty_viajeros"] = pd.to_numeric(
        df_viajeros["qty_viajeros"]
            .str.replace(r"\.", "", regex=True)
            .str.replace(",", ".", regex=False),
        errors="coerce"
    )

if df_pernoctaciones["qty_viajeros"].dtype == "object":
    df_pernoctaciones["qty_viajeros"] = pd.to_numeric(
        df_pernoctaciones["qty_viajeros"]
            .str.replace(r"\.", "", regex=True)
            .str.replace(",", ".", regex=False),
        errors="coerce"
    )

In [10]:
def limpiar_ccaa(texto):
    if pd.isna(texto):
        return texto

    # quitar códigos tipo "01 "
    texto = re.sub(r"^\d+\s+", "", texto)

    # normalizar
    texto = texto.strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")

    return texto

def ajustar_ccaa(texto):
    if pd.isna(texto):
        return texto

    if "balears" in texto:
        return "Islas Baleares"
    elif "valenciana" in texto:
        return "Comunidad Valenciana"
    elif "madrid" in texto:
        return "Madrid"
    elif "cataluna" in texto:
        return "Cataluña"
    elif "andalucia" in texto:
        return "Andalucia"
    
    return texto

df_viajeros["ccaa"] = df_viajeros["ccaa"].apply(limpiar_ccaa).apply(ajustar_ccaa)
df_pernoctaciones["ccaa"] = df_pernoctaciones["ccaa"].apply(limpiar_ccaa).apply(ajustar_ccaa)

In [11]:
df_pernoctaciones

,ccaa,pais_origen,mes,qty_viajeros,año
0,Andalucia,Residentes en España,Enero,902311.0,2015
1,Andalucia,Residentes en España,Febrero,1255482.0,2015
2,Andalucia,Residentes en España,Marzo,1644221.0,2015
3,Andalucia,Residentes en España,Abril,1900705.0,2015
4,Andalucia,Residentes en España,Mayo,1711168.0,2015
...,...,...,...,...,...
6895,Madrid,Países africanos,Agosto,34269.0,2019
6896,Madrid,Países africanos,Septiembre,33728.0,2019
6897,Madrid,Países africanos,Octubre,37419.0,2019
6898,Madrid,Países africanos,Noviembre,47053.0,2019


In [12]:
df_viajeros

,ccaa,pais_origen,mes,qty_viajeros,año
0,Andalucia,Residentes en España,Enero,435583.0,2015
1,Andalucia,Residentes en España,Febrero,592854.0,2015
2,Andalucia,Residentes en España,Marzo,671180.0,2015
3,Andalucia,Residentes en España,Abril,769073.0,2015
4,Andalucia,Residentes en España,Mayo,812986.0,2015
...,...,...,...,...,...
6895,Madrid,Países africanos,Agosto,11537.0,2019
6896,Madrid,Países africanos,Septiembre,10427.0,2019
6897,Madrid,Países africanos,Octubre,12293.0,2019
6898,Madrid,Países africanos,Noviembre,12779.0,2019


In [ ]:
# Generación del CSV bloqueada (el archivo ya se encuentra en el directorio del proyecto).
#df_pernoctaciones.to_csv('../../data/clean_pernoctaciones_07_04_2026.csv', index=False)

#df_viajeros.to_csv('../../data/clean_viajeros_07_04_2026.csv', index=False)